In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [ ]:
# Set the style of seaborn plots
sns.set_theme(style='darkgrid')

# Define full features, policy names, titles
featnames = {'lexdiv': 'Lexical Diversity',
            'sentcomp': 'Sentiment',
            'sim': 'Topical Similarity',
            'smog': 'Readability'}
polnames = {'chron': 'Chronological',
            'least-neg-abs': 'Rev. Downvotes',
            'neg-abs': 'Downvotes',
            'pos-abs': 'Upvotes',
            'pos-rel': 'Relative Votes',
            'rev-chron': 'Rev. Chronological',
            'pred-nb': 'Pred. Upvotes (NBR)',
            'pred-xgb': 'Pred. Upvotes (XGB)',            
            'pin-pred-lr': 'Pred. Editors\' Picks (LR)',
            'pin-pred-xgb': 'Pred. Editors\' Picks (XGB)',
            'random': 'Random',
            'policy_chron': 'Chronological',
            'policy_least-neg-abs': 'Rev. Downvotes',
            'policy_neg-abs': 'Downvotes',
            'policy_pos-abs': 'Upvotes',
            'policy_pos-rel': 'Relative Votes',
            'policy_rev-chron': 'Rev. Chronological',
            'policy_random': 'Random',
            'policy_pin-pred-lr': 'Pred. Editors\' Picks (LR)',
            'policy_pin-pred-xgb': 'Pred. Editors\' Picks (XGB)',
            'policy_pred-nb': 'Pred. Upvotes (NBR)',
            'policy_pred-xgb': 'Pred. Upvotes (XGB)',
            'replies_rh': 'Replies Hidden',
            'replies_rt': 'Reply Trees Shown',
            'replies_rl': 'Replies Loose',
            'replies hidden': 'Replies Hidden',
            'reply trees': 'Reply Trees Shown',
            'replies loose': 'Replies Loose',
            'pinned': 'Editors\' Picks Pinned',
            'not_pinned': 'Editors\' Picks Not Pinned'}
titles = {'10': 'First 10 Comments','N': 'Full Comment Discussion'}
repl = {'rh': 'Hidden',
        'rt': 'Trees',
        'rl': 'Loose'}


# Function to parse policy elements from policy name
def policynameparse(x):
    if x.startswith('policy_'):
        x = x[7:]

    spl = x.split('_')
    policy = polnames[spl[0]]
    replies = repl[spl[-1]]
    pinned = 'Pinned' if 'pinned' in x else 'Not Pinned'

    return policy, replies, pinned

# Dictionary to map feature names to colors
color_dict = dict(zip(['Lexical Diversity', 'Sentiment', 'Topical Similarity',
                      'Readability'], [sns.color_palette("colorblind")[:4]]))

In [ ]:
q_df = pd.read_csv('data/q_df.csv')
q_df['n'] = q_df['n'].astype(str)
q_df

In [ ]:
fig, axs = plt.subplots(4, 2, figsize=(8, 5), sharex=True)
for nc, n in enumerate(['10', 'N']):
    for nf, feat in enumerate(['lexdiv', 'sentcomp', 'sim', 'smog']):
        ax = axs[nf, nc]
        if nf == 0:
            ax.text(0, -2, titles[n], ha='center')

        fn_df = q_df[(q_df['feature']==feat) & (q_df['n']==n)].copy()
        fn_df['sorting policy'] = fn_df['sorting policy'].apply(policynameparse)
        fn_df['sorting policy'] = fn_df['sorting policy'].apply(lambda x: '_'.join(x))
        fg = fn_df.groupby('sorting policy')['value'].mean()
        order = fg.sort_values(ascending=False).index
        subset = [order[0], 'Rev. Chronological_Trees_Pinned', order[-1]]
        fn_df = fn_df[fn_df['sorting policy'].isin(subset)]
        fn_df = fn_df.set_index('sorting policy').loc[subset].reset_index()


        # box and whisker plot by sorting policy
        sns.violinplot(data=fn_df, y='sorting policy', x='value', hue='sorting policy',
                       order=subset, inner='quart', linewidth=0.5, cut=0,
                     ax=ax)
        # add axis gridlines at -0.5, 0.5
        ax.set_xticks([-1, -0.5, 0, 0.5, 1])
        ax.set_xticklabels([-1, -0.5, 0, 0.5, 1])
        ax.xaxis.grid(True)

        # add a dot for mean value
        ax.scatter(fg.loc[list(subset)],
                   range(len(subset)), color='black', s=5)
        
        ax.set_yticklabels([])

        scale = 1
        cellw = 1
        cellx = 1.1
        celly = 0.65
        if nf==0:
                ax.text(-2.2, -1.5*scale, 'Policy', fontsize=10, style='italic',
                        ha='center', va='center')
                ax.text(-(cellw+cellx+celly), -0.8*scale, 'Ordering', fontsize=10, style='italic',
                        ha='right', va='center')
                ax.text(-(cellw+cellx), -0.8*scale, 'Replies', fontsize=10, style='italic',
                        ha='right', va='center')
                ax.text(-cellw, -0.8*scale, 'Editors\' Picks', fontsize=10, style='italic',
                        ha='right', va='center')
        for ix, p in enumerate(subset):
            if p == 'Rev. Chronological_Trees_Pinned':
                weight = 'bold'
            else:
                weight = 'normal'

            ax.text(-(cellw+cellx+celly), ix*scale, p.split('_')[0], fontsize=10, weight=weight,
                    ha='right', va='center')
            ax.text(-(cellw+cellx), ix*scale, p.split('_')[1], fontsize=10, weight=weight,
                     ha='right', va='center')
            ax.text(-cellw, ix*scale, p.split('_')[2], fontsize=10, weight=weight,
                     ha='right', va='center')

        ax.set_title(featnames[feat], fontsize=10)
        # make y label invisible
        ax.set_ylabel('')
        ax.set_xlabel('')
        
        ax.set_xlim(-1, 1)
    ax.set_xlabel('$\Phi_{\\text{%s}}$' %n)
fig.suptitle('Distribution of FORUM Scores for Best/Default/Worst Policies Over All Discussions')
fig.subplots_adjust(wspace=2, hspace=0.5, top=0.84)
# fig.tight_layout()
plt.savefig('figs/qdist_best_default_worst.pdf', bbox_inches='tight')
plt.savefig('figs/qdist_best_default_worst.svg', bbox_inches='tight')
plt.savefig('figs/qdist_best_default_worst.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 5))

n = '10'
feat = 'lexdiv'

fn_df = q_df[(q_df['feature']==feat) & (q_df['n']==n)].copy()
fn_df['sorting policy'] = fn_df['sorting policy'].apply(policynameparse)
fn_df['sorting policy'] = fn_df['sorting policy'].apply(lambda x: '_'.join(x))
fg = fn_df.groupby('sorting policy')['value'].mean()
order = fg.sort_values(ascending=False).index
subset = [order[0], 'Rev. Chronological_Trees_Pinned', order[-1]]
fn_df = fn_df[fn_df['sorting policy'].isin(subset)]
fn_df = fn_df.set_index('sorting policy').loc[subset].reset_index()


# box and whisker plot by sorting policy
sns.violinplot(data=fn_df, y='sorting policy', x='value', hue='sorting policy',
                order=subset, inner='quart', linewidth=0.5, cut=0,
                ax=ax)
# add axis gridlines at -0.5, 0.5
ax.set_xticks([-1, -0.5, 0, 0.5, 1])
ax.set_xticklabels([-1, -0.5, 0, 0.5, 1])
ax.xaxis.grid(True)

# add a dot for mean value
ax.scatter(fg.loc[list(subset)],
            range(len(subset)), color='black', s=5)

ax.set_yticklabels([])

scale = 1
cellw = 1.1
cellx = 0.4
celly = 0.25
ax.text(-(cellw+cellx+celly), -0.6*scale, 'Ordering', fontsize=10, style='italic',
        ha='right', va='center')
ax.text(-(cellw+cellx), -0.6*scale, 'Replies', fontsize=10, style='italic',
        ha='right', va='center')
ax.text(-cellw, -0.6*scale, 'Editors\' Picks', fontsize=10, style='italic',
        ha='right', va='center')
for ix, p in enumerate(subset):
    if p == 'Rev. Chronological_Trees_Pinned':
        weight = 'bold'
    else:
        weight = 'normal'

    ax.text(-(cellw+cellx+celly), ix*scale, p.split('_')[0], fontsize=10, weight=weight,
            ha='right', va='center')
    ax.text(-(cellw+cellx), ix*scale, p.split('_')[1], fontsize=10, weight=weight,
                ha='right', va='center')
    ax.text(-cellw, ix*scale, p.split('_')[2], fontsize=10, weight=weight,
                ha='right', va='center')

ax.set_title(featnames[feat], fontsize=10)
# make y label invisible
ax.set_ylabel('')
ax.set_xlabel('')

ax.set_xlim(-1, 1)
ax.set_xlabel('$\Phi_{\\text{%s}}$' %n)
fig.suptitle('Distribution of FORUM Scores for Best/Default/Worst Policies\nOver First 10 Comments')
# add space to suptitle
fig.subplots_adjust(top=0.81)
fig.subplots_adjust(bottom=0.23)
fig.text(x=0.3, y=0.05, s='The "Pred. Upvotes (NBR), Replies Loose, Editors\' Picks Not Pinned" algorithm prioritises lexically diverse comments the most.\nThe "Rev. Downvotes, Reply Trees, Editors\' Picks Pinned" algorithm prioritises lexically diverse comments the least, worse than at random.\nThe "Rev. Chronological, Reply Trees, Editors\' Picks Pinned" (default) algorithm prioritises lexically diverse comments slightly better than random.', ha='center', va='center', fontsize=10)
         
fig.savefig('figs/FORUM_tweet_example.png', bbox_inches='tight')


In [ ]:
for n in ['10', 'N']:
    fig, axs = plt.subplots(1, 4, figsize=(20, 12))
    for nf, feat in enumerate(['lexdiv', 'sentcomp', 'sim', 'smog']):
        ax = axs[nf]

        fn_df = q_df[(q_df['feature']==feat) & (q_df['n']==n)].copy()
        fn_df['sorting policy'] = fn_df['sorting policy'].apply(policynameparse)
        fn_df['sorting policy'] = fn_df['sorting policy'].apply(lambda x: '_'.join(x))
        fg = fn_df.groupby('sorting policy')['value'].mean()
        order = fg.sort_values(ascending=False).index
        fn_df = fn_df.set_index('sorting policy').loc[order].reset_index()

        # box and whisker plot by sorting policy
        sns.violinplot(data=fn_df, y='sorting policy', x='value', hue='sorting policy',
                       order=order, inner='quart', linewidth=0.5, cut=0, palette=sns.color_palette("Spectral", n_colors=66),
                     ax=ax)
        # add axis gridlines at -0.5, 0.5
        ax.set_xticks([-1, -0.5, 0, 0.5, 1])
        ax.set_xticklabels([-1, -0.5, 0, 0.5, 1])
        ax.xaxis.grid(True)

        # add a dot for mean value
        ax.scatter(fg.loc[order],
                   range(len(order)), color='black', s=5)
        
        ax.set_yticklabels([])

        scale = 1
        cellw = 1
        cellx = 1.075
        celly = 1.7
        ax.text(-2.3, -2*scale, 'Policy', fontsize=10, style='italic',
                ha='center', va='center')
        ax.text(-(cellw+celly), -scale, 'Ordering', fontsize=10, style='italic',
                ha='right', va='center')
        ax.text(-(cellw+cellx), -scale, 'Replies', fontsize=10, style='italic',
                ha='right', va='center')
        ax.text(-cellw, -scale, 'Editors\' Picks', fontsize=10, style='italic',
                ha='right', va='center')
        for ix, p in enumerate(order):
            if p == ('Rev. Chronological', 'Trees', 'Pinned'):
                weight = 'bold'
            else:
                weight = 'normal'

            ax.text(-(cellw+celly), ix*scale, p.split('_')[0], fontsize=10, weight=weight,
                    ha='right', va='center')
            ax.text(-(cellw+cellx), ix*scale, p.split('_')[1], fontsize=10, weight=weight,
                     ha='right', va='center')
            ax.text(-cellw, ix*scale, p.split('_')[2], fontsize=10, weight=weight,
                     ha='right', va='center')

        ax.set_title(featnames[feat])
        # make y label invisible
        ax.set_ylabel('')
        
        ax.set_xlabel('$\Phi_{\\text{%s}}$' %n)
        ax.set_xlim(-1, 1)
        ax.set_ylim(len(order), -2.5)
    fig.suptitle(f'Distribution of FORUM Score for Every Policy in Each Feature Over All Discussions, {titles[n]}')
    fig.subplots_adjust(wspace=1.9, top=0.93)
    plt.savefig(f'figs/qdist_all_{n}.pdf', bbox_inches='tight')
    plt.savefig(f'figs/qdist_all_{n}.svg', bbox_inches='tight')
    plt.show()
